## Pods

In this notebook, we look at Kubernetes pods.

- Make sure you have a Minikube Kubernetes cluster running, e.g. `minukube start`.
- Also make sure you have installed the `kubectl` tool on your computer.

## Create a Pod using the imperative approach

- A Pod is the smallest unit of work in a kubernetes cluster.
  - In contains one or more Containers.
  - Usually there is only one Container per Pod.
- The command `kubectl run [PodName] --image [ImageName]`:
  - Creates a Pod with name [PodName].
  - Running a container based on the Docker image `[ImageName]`.
- In this case the Pod is called `mynginx` and is running a container based on the `nginx` image.

In [1]:
!kubectl run mynginx --image nginx

pod/mynginx created


## List Pods

- The command `kubectl get pods` lists Pods running in the cluster.

The information about a Pod is:
- `NAME` : The unique name of the Pod (`mynginx`).
- `READY` : The number of Containers (out of the total number of Containers) in the Pod that are ready and running (`1/1`).
- `STATUS` : The status of the Pod (`Running`).
- `RESTARTS` : How many times the Pod has restarted (`0`).
- `AGE` : The age of the Pod, since the creation of the Pod (in seconds).

The `STATUS` of a Pod can be:
- `Pending` : The Pod has been created but its containers are not yet running.
- `Running` : The Pod is running  with at least one container in a running state.
- `Succeeded` : All containers in the Pod have terminated successfully (with an exit status of 0).
- `Failed` : One or more containers in the Pod have terminated with a non-zero exit status (error).
- `Unknown` : The state of the Pod cannot be determined and is typically temporary due to Node failures or network issues.
- `CrashLoopBackOff` : One or more containers in the Pod started, then crashed, and restarted, over and over again.

In [3]:
#!kubectl get po
!kubectl get pods

NAME      READY   STATUS    RESTARTS   AGE
mynginx   1/1     Running   0          17s


## List Pods (with additional information)

- The `-o wide` option displays additional information.

The additonal information about a Pod is:
- `IP` : The Pod's IP address (within the cluster's internal/private network).
- `NODE` : The Node where the Pod is currently running.
- `NOMINATED NODE` : The preferred Node to run the Pod on, identified by the kube-scheduler based on the Pod’s resource requirements (CPU, RAM, etc.).
- `READINESS GATES` : The conditions that must be satisfied for the Pod to be considered ready.

In [4]:
#!kubectl get po -o wide
!kubectl get pods -o wide

NAME      READY   STATUS    RESTARTS   AGE   IP           NODE       NOMINATED NODE   READINESS GATES
mynginx   1/1     Running   0          24s   10.244.0.3   minikube   <none>           <none>


## List a specific Pod

- The command `kubectl get pod [PodName]` lists a Pod with name `[PodName]`.
- Here we are listing the Pod with name `mynginx`.

In [5]:
#!kubectl get po mynginx
#!kubectl get pod mynginx
#!kubectl get po mynginx -o wide
!kubectl get pod mynginx -o wide

NAME      READY   STATUS    RESTARTS   AGE   IP           NODE       NOMINATED NODE   READINESS GATES
mynginx   1/1     Running   0          37s   10.244.0.3   minikube   <none>           <none>


## Get the YAML for a Pod

- The command `kubectl get pod [PodName] -o yaml` returns the YAML for a Pod with name `[PodName]`.
  - The YAML also contains status information about the Pod.
- Here we are getting the YAML for the Pod with name `mynginx`.

In [6]:
#!kubectl get po mynginx -o yaml
!kubectl get pod mynginx -o yaml

apiVersion: v1
kind: Pod
metadata:
  creationTimestamp: "2026-03-14T19:14:05Z"
  generation: 1
  labels:
    run: mynginx
  name: mynginx
  namespace: default
  resourceVersion: "1424"
  uid: b0a2855f-8f98-4bcf-87ca-8665188ae6f7
spec:
  containers:
  - image: nginx
    imagePullPolicy: Always
    name: mynginx
    resources: {}
    terminationMessagePath: /dev/termination-log
    terminationMessagePolicy: File
    volumeMounts:
    - mountPath: /var/run/secrets/kubernetes.io/serviceaccount
      name: kube-api-access-sj65f
      readOnly: true
  dnsPolicy: ClusterFirst
  enableServiceLinks: true
  nodeName: minikube
  preemptionPolicy: PreemptLowerPriority
  priority: 0
  restartPolicy: Always
  schedulerName: default-scheduler
  securityContext: {}
  serviceAccount: default
  serviceAccountName: default
  terminationGracePeriodSeconds: 30
  tolerations:
  - effect: NoExecute
    key: node.kubernetes.io/not-ready
    operator: Exists
    tolerationSeconds: 300
  - effect: NoExecute
   

## Describe a Pod

- The command `kubectl describe pod [PodName]` returns detailed information about a Pod with name `[PodName]`.
  - The description of a Pod contains the same information as the YAML listing above, but also includes additional information.
  - At the very bottom of the description, important Pod `Events` are given, e.g.:

    ```bash
    Events:
      Type    Reason     Age   From               Message
      ----    ------     ----  ----               -------
      Normal  Scheduled  51s   default-scheduler  Successfully assigned default/mynginx to minikube
      Normal  Pulling    50s   kubelet            spec.containers{mynginx}: Pulling image "nginx"
      Normal  Pulled     44s   kubelet            spec.containers{mynginx}: Successfully pulled image "nginx" in 6.259s.
      Normal  Created    43s   kubelet            spec.containers{mynginx}: Container created
      Normal  Started    43s   kubelet            spec.containers{mynginx}: Container started
    ```

- Here we are describing the Pod named `mynginx`.

In [8]:
#!kubectl describe po mynginx
!kubectl describe pod mynginx

Name:             mynginx
Namespace:        default
Priority:         0
Service Account:  default
Node:             minikube/192.168.49.2
Start Time:       Sat, 14 Mar 2026 20:14:05 +0100
Labels:           run=mynginx
Annotations:      <none>
Status:           Running
IP:               10.244.0.3
IPs:
  IP:  10.244.0.3
Containers:
  mynginx:
    Container ID:   docker://22d9267b4dbd6a33f121c2f6341a4ce0e56568ff4fe83ff950f264c8cbc748e9
    Image:          nginx
    Image ID:       docker-pullable://nginx@sha256:bc45d248c4e1d1709321de61566eb2b64d4f0e32765239d66573666be7f13349
    Port:           <none>
    Host Port:      <none>
    State:          Running
      Started:      Sat, 14 Mar 2026 20:14:13 +0100
    Ready:          True
    Restart Count:  0
    Environment:    <none>
    Mounts:
      /var/run/secrets/kubernetes.io/serviceaccount from kube-api-access-sj65f (ro)
Conditions:
  Type                        Status
  PodReadyToStartContainers   True 
  Initialized                 T

## Show the Logs for a Pod

- The command `kubectl logs [PodName]` shows the logs for the Pod named `[PodName]`.
  - The logs contain anything written to the standard output and error streams (`stdout`, `stderr`) by a Pod's containers.
- In this case the logs are displayed for the Pod `mynginx`.

In [9]:
!kubectl logs mynginx

/docker-entrypoint.sh: /docker-entrypoint.d/ is not empty, will attempt to perform configuration
/docker-entrypoint.sh: Looking for shell scripts in /docker-entrypoint.d/
/docker-entrypoint.sh: Launching /docker-entrypoint.d/10-listen-on-ipv6-by-default.sh
10-listen-on-ipv6-by-default.sh: info: Getting the checksum of /etc/nginx/conf.d/default.conf
10-listen-on-ipv6-by-default.sh: info: Enabled listen on IPv6 in /etc/nginx/conf.d/default.conf
/docker-entrypoint.sh: Sourcing /docker-entrypoint.d/15-local-resolvers.envsh
/docker-entrypoint.sh: Launching /docker-entrypoint.d/20-envsubst-on-templates.sh
/docker-entrypoint.sh: Launching /docker-entrypoint.d/30-tune-worker-processes.sh
/docker-entrypoint.sh: Configuration complete; ready for start up
2026/03/14 19:14:13 [notice] 1#1: using the "epoll" event method
2026/03/14 19:14:13 [notice] 1#1: nginx/1.29.6
2026/03/14 19:14:13 [notice] 1#1: built by gcc 14.2.0 (Debian 14.2.0-19) 
2026/03/14 19:14:13 [notice] 1#1: OS: Linux 6.17.0-14-gener

## Delete a Pod

- The command `kubectl delete pod [PodName]`deletes the Pod with the name `[PodName]`:
  - In this case the Pod `mynginx` is deleted.

In [10]:
#!kubectl delete po mynginx
!kubectl delete pod mynginx

pod "mynginx" deleted from default namespace


## List Pods

- We see the Pod has been deleted.

In [11]:
#!kubectl get po
!kubectl get pods

No resources found in default namespace.


## Create a pod running a BusyBox container, with an interactive shell

### Run these command in a terminal (won't work from a notebook cell)

```bash
# Create a pod running a BusyBox container, in interactive mode (i.e. attach an interactive shell)
kubectl run mybox --image=busybox -it -- /bin/sh

# --- Output ---
# if you don't see a command prompt, try pressing enter.
# / # 
# --------------

# List folders in the container
ls

# --- Output ---
# bin    dev    etc    home   lib    lib64  proc   root   sys    tmp    usr    var
# / #
# --------------

# Issue a command in the container
# This encodes (NOT encrypts) the text 'A Secret' in base64 encoding format
echo -n 'A Secret' | base64

# --- Output ---
# QSBTZWNyZXQ=
# / #
# --------------

# Issue another command in the container
# This decodes (NOT decrypts) the base64-encoded text 'QSBTZWNyZXQ='
echo -n 'QSBTZWNyZXQ=' | base64 -d

# --- Output ---
# A Secret
# / #
# --------------

# Exit the interactive shell in the container
exit

# --- Output ---
# Session ended, resume using 'kubectl attach mybox -c mybox -i -t' command when the pod is running
# --------------

```

## Delete a Pod immediately

- The command `kubectl delete pod [PodName]` deletes as Pod named `[PodName]` (in this case `mybox`).
  - The `--grace-period 0` option starts deleting the Pod immediately (the default grace period is 30 seconds).
  - The `--force` option deletes a Pod immediately.

In [12]:
!kubectl delete pod mybox --grace-period 0
#!kubectl delete pod mybox --grace-period 0 --force
#!kubectl delete pod mybox

pod "mybox" deleted from default namespace


## Create a Pod using the declarative approach

- The command `kubectl apply -f [PodYAMLFile]` creates (or updates) the Pod defined in the YAML file `[PodYAMLFile]`.
  - The `-f` option gives the path to the YAML file.
  - Alternatively the command `kubectl create -f [PodYAMLFile]` can be used to create a Pod (but not to update it).

- The Pod definition is in the YAML file `manifests/myapp.yaml`

In [13]:
#!kubectl create -f manifests/myapp.yaml
!kubectl apply -f manifests/myapp.yaml

pod/myapp-pod created


## Let's look at the Pod's YAML file

The YAML file contains:

```bash
apiVersion: v1          # the version of the API used to define the resource
kind: Pod               # the resource type being defined, i.e. Pod
metadata:               # the map of metadata for the Pod starts here
  name: myapp-pod         # the name of the Pod
  labels:                 # the map of labels (key-value pairs) to define for the Pod starts here
    app: myapp              # a label with name "app" and value "myapp"
    type: front-end         # a label with name "type" and value "front-end"
spec:                   # the Pod's specification starts here
  containers:                 # the list of containers to run in the Pod starts here (in this case only one)
  - name: nginx-container       # the name of the container
    image: nginx                # the image to base the container on    
    ports:                      # the list of ports the container will listen on starts here (in this case only one)
    - containerPort: 80           # the port's number
      name: http                  # the port's name
      protocol: TCP               # the port's protocol
    env:                        # the list of container environment variables to set starts here (in this case only one)
    - name: DBCON                 # the environment variable's name
      value: myconnectionstring   # the environment variable's value
    resources:                  # the map of resource specifications for the container starts here
      requests:                   # the map of requested resources for the container starts here
        cpu: 100m                   # requested CPU in "millicores" (1m is 1/1000 of a CPU core, 1000m is 1 CPU core)
        memory: 128Mi               # requested RAM in "mebibytes" (1Mi is 1 048 576 bytes, 1Ki is 1024 bytes)
      limits:                     # the map of maximum resources for the container starts here
        cpu: 250m                   # maximum CPU in "millicores" (1m is 1/1000 of a CPU core, 1000m is 1 CPU core)
        memory: 256Mi               # maximum RAM in "mebibytes" (1Mi is 1 048 576 bytes, 1Ki is 1024 bytes)
```

**Note:**
- If `resources` (`cpu` and `memory`) aren't specified for a container:
  - The container is free to use an unlimited amount of CPU and RAM on the node it is running on.
- `requests` is the amount of resources (`cpu` and `memory`) the container is asking for.
- `limmits` is the maximum amount of resources (`cpu` and `memory`) the container is asking to boost to if needed.
- In order to use a `Horizontal Pod Autoscaler (HPA)`, `resources` must be specified for a container.

In [14]:
#!type manifests\myapp.yaml # Use this on Linux/Mac
!cat manifests/myapp.yaml # Use this on Linux/Mac

apiVersion: v1
kind: Pod
metadata:
  name: myapp-pod
  labels:
    app: myapp
    type: front-end
spec:
  containers:
  - name: nginx-container
    image: nginx
    resources:
      requests:
        cpu: 100m
        memory: 128Mi
      limits:
        cpu: 250m
        memory: 256Mi    
    ports:
    - containerPort: 80
      name: http
      protocol: TCP
    env:
    - name: DBCON
      value: myconnectionstring

## List Pods

- We see the Pod is running.

In [15]:
#!kubectl get po -o wide
!kubectl get pods -o wide

NAME        READY   STATUS    RESTARTS   AGE   IP           NODE       NOMINATED NODE   READINESS GATES
myapp-pod   1/1     Running   0          13s   10.244.0.5   minikube   <none>           <none>


## Attach an interative terminal to the pod (container)

### Run these command in a terminal (won't work from a notebook cell)

```bash
# Attach to an interactive terminal in the pod (container)
kubectl exec -it myapp-pod -- bash

# --- Output ---
# root@myapp-pod:/#
# --------------

# Print the DBCON environment variable (set in the YAML file) in the container
echo $DBCON

# --- Output ---
# myconnectionstring
# root@myapp-pod:/#
# --------------

# Get the path to the current directory.
# 'pwd' stands for "Print Working Directory".
# In this case "/" means we're in the root directory of the Linux file system.
pwd

# --- Output ---
# /
# root@myapp-pod:/#
# --------------

# List files and folders in the current directory
# The command is 'ls'
# The 'l' option stands for "long listing format", i.e. also lists file permissions
# The 'a' option stands for "all", also lists hidden files and folders, that start with a "dot" .
ls -al

# --- Output ---
# total 72
# drwxr-xr-x   1 root root 4096 Feb 15 09:27 .
# drwxr-xr-x   1 root root 4096 Feb 15 09:27 ..
# -rwxr-xr-x   1 root root    0 Feb 15 09:27 .dockerenv
# lrwxrwxrwx   1 root root    7 Jan  2 12:35 bin -> usr/bin
# drwxr-xr-x   2 root root 4096 Jan  2 12:35 boot
# drwxr-xr-x   5 root root  360 Feb 15 09:27 dev
# drwxr-xr-x   1 root root 4096 Feb  4 23:53 docker-entrypoint.d
# -rwxr-xr-x   1 root root 1620 Feb  4 23:52 docker-entrypoint.sh
# drwxr-xr-x   1 root root 4096 Feb 15 09:27 etc
# drwxr-xr-x   2 root root 4096 Jan  2 12:35 home
# lrwxrwxrwx   1 root root    7 Jan  2 12:35 lib -> usr/lib
# lrwxrwxrwx   1 root root    9 Jan  2 12:35 lib64 -> usr/lib64
# drwxr-xr-x   2 root root 4096 Feb  2 00:00 media
# drwxr-xr-x   2 root root 4096 Feb  2 00:00 mnt
# drwxr-xr-x   2 root root 4096 Feb  2 00:00 opt
# dr-xr-xr-x 729 root root    0 Feb 15 09:27 proc
# drwx------   2 root root 4096 Feb  2 00:00 root
# drwxr-xr-x   1 root root 4096 Feb 15 09:27 run
# lrwxrwxrwx   1 root root    8 Jan  2 12:35 sbin -> usr/sbin
# drwxr-xr-x   2 root root 4096 Feb  2 00:00 srv
# dr-xr-xr-x  13 root root    0 Feb 15 09:34 sys
# drwxrwxrwt   2 root root 4096 Feb  2 00:00 tmp
# drwxr-xr-x   1 root root 4096 Feb  2 00:00 usr
# drwxr-xr-x   1 root root 4096 Feb  2 00:00 var
# root@myapp-pod:/#
# --------------

# Detach from the container
exit

# --- Output ---
# exit
# --------------
```

## Delete the pod

- The command `kubectl delete -f [PodYAMLFile]` deletes the Pod defined in the YAML file `[PodYAMLFile]`.
  - In this case the Pod defined in the YAML file `myapp.yaml` is deleted.

In [16]:
!kubectl delete -f manifests/myapp.yaml

pod "myapp-pod" deleted from default namespace


## List Pods

- We see that the Pod has been deleted.

In [17]:
#!kubectl get po
!kubectl get pods

No resources found in default namespace.
